# 00 · R1→R0 전이 비교의 공통 준비
A: R0 추가 학습 / B: R1 인코더 + 새 R0 출력층 / C: 기존 R1 동결. 이 노트북은 입력 검증과 공통 피드백 대리 로그를 만듭니다.

**실제 사용자 만족을 생성하지 않습니다.** 공개 benchmark 난이도 라벨을 상향 교정 대리값으로 쓰며, 모든 비교군이 같은 피드백을 받습니다. 상세 계약은 `docs/ml/R1_TO_R0_TRANSFER_EXPERIMENT.md`에 있습니다.

## 공통 준비
개인 `alexyoo0506@gmail.com` Colab Pro와 해당 계정의 Drive를 사용합니다. 사용자가 Drive 5TB 보유를 확인했으므로 저장용량 확인·입력 절차는 생략합니다. Drive 연결 시 해당 계정을 선택합니다. GPU 작업은 T4 런타임을 선택하고, Drive 연결은 Google 화면에서 직접 승인합니다. GCP·Vertex 제출 또는 GCS 업로드 코드는 없습니다.

현재 파일은 아직 원격 main에 게시되지 않았으므로 main을 clone해서 실행하지 않습니다. 제공한 실행 패키지의 `transfer-experiments` 폴더를 개인 Drive의 `MyDrive/LLM_Router/`에 업로드하세요. 경로·SHA·데이터·source pin이 모두 채워져 있으므로 수정 없이 전체 실행합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

BASE = Path("/content/drive/MyDrive/LLM_Router/transfer-experiments")
SOURCE_ZIP = BASE / "source" / "transfer-source.zip"
EXPECTED_SOURCE_SHA256 = SOURCE_ZIP.with_suffix(".sha256").read_text().split()[0]
RUN_ID = "r1-to-r0-transfer-v2-run001"  # A/B/C 모두 같은 ID. 새 계획에는 새 ID
RUN = BASE / "runs" / RUN_ID
INPUTS = BASE / "inputs" / "experiment-A"
PINS = INPUTS / "source_pins.json"
DATA = INPUTS / "data"
LOG = INPUTS / "feedback-log.parquet"

assert len(EXPECTED_SOURCE_SHA256) == 64, "소스 ZIP의 검증된 SHA-256을 입력하세요"
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_SOURCE_SHA256
REPO = Path("/content/router-transfer-source")
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, expected in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == expected, name
EXPERIMENT = REPO / "ml/notebooks/r1_to_r0_transfer"
HELPER = EXPERIMENT / "experiment.py"
CONFIG = EXPERIMENT / "experiment_config.json"
ENV = dict(os.environ, PYTHONPATH=str(REPO / "ml/src"))
def run(*args):
    subprocess.run([sys.executable, str(HELPER), *map(str, args)], env=ENV, check=True)
print("Shared run directory:", RUN)


## 패키지 설치
Colab 메뉴 **런타임 → 런타임 유형 변경 → 런타임 버전 2026.07**(Python 3.12.13), GPU T4를 선택합니다. 기본 최신 버전은 사용하지 않습니다. Python 3.11/3.12와 CUDA 12.4용 PyTorch 2.5.1을 사용합니다. 설치 후 이미 import한 패키지 버전이 다르면 Colab 세션을 재시작하고 경로 설정 셀부터 다시 실행하세요. 학습 helper가 런타임 계약 불일치를 검사합니다.

In [ ]:
assert sys.version_info[:2] in {(3, 11), (3, 12)}, (f"현재 Python {sys.version.split()[0]}. Colab 런타임 → 런타임 유형 변경 → 런타임 버전을 2026.07(Python 3.12)로 바꾸고 처음부터 모두 실행하세요. T4는 유지합니다.")
# Text-only model: remove preinstalled optional packages incompatible with pinned torch.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(EXPERIMENT / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO / "ml")], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요"
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)


## 준비된 입력
기존 봉인 분할에서 만든 6개 데이터 파일과 실제 원본의 SHA·모델 경로가 이미 설정되어 있습니다. 새 데이터 분할·초기학습을 하지 않습니다. 아래 셀이 개인 Drive의 기존 R0/R1 archive를 검증하고 모델을 복구합니다.


In [ ]:
assert PINS.is_file(), f"실행 패키지 폴더를 Drive에 업로드하세요: {PINS}"
subprocess.run([sys.executable, str(EXPERIMENT / "prepare_models.py"), "--inputs", str(INPUTS)], env=ENV, check=True)


## 공통 logger 추론과 준비
아직 로그가 없을 때만 원본 R0로 feedback 분할을 8192 토큰에서 추론합니다. 같은 로그가 이미 있으면 sidecar 검증 후 재사용합니다. filter 전 여섯 window를 나누며 각 window에 32개 이상의 상향 교정 후보가 없으면 준비가 중단됩니다. 임의로 가짜 행을 보충하지 않습니다.

In [ ]:
if not LOG.exists():
    run("build-feedback-log", "--config", CONFIG, "--pins", PINS, "--data-dir", DATA, "--output-file", LOG)
if not RUN.exists():
    run("prepare", "--config", CONFIG, "--pins", PINS, "--data-dir", DATA, "--feedback-log", LOG, "--output-dir", RUN)
else:
    import importlib.util
    spec = importlib.util.spec_from_file_location("transfer_helper", HELPER)
    helper = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(helper)
    plan = helper.load_plan(RUN)
    cfg, pins, _ = helper.load_inputs(CONFIG, PINS, DATA)
    assert plan["config"] == cfg and plan["pins"] == pins, "다른 입력/설정이면 새 RUN_ID를 사용하세요"
    assert plan["feedback_receipt"]["log_sha256"] == helper.sha(LOG)
    print("기존 RUN 계약과 입력 검증 완료")
print(json.loads((RUN / "plan.json").read_text())["phase_train_rows"])
